# Algebra for Probabilistic Knowledge Processing — Exercises with EQL

**Course:** AI – Knowledge Acquisition and Knowledge Representation, Summer Term 2026  
**Companion to:** Lecture 2 (Algebra)

---

## Story: The RoboFleet Warehouse

You are the lead engineer at **RoboFleet**, a warehouse automation company in Bremen. Your warehouse has a fleet of robots, each assigned to zones with different priority levels. Robots carry sensors that stream position data as they navigate.

Your job: use **queries** to reason about which robots are where, which assignments are valid, and which sensor readings fall within safe operating regions. Along the way, you will discover that these queries correspond directly to the algebraic structures from the lecture. Important concepts are measurable spaces, product sigma algebras, set operations, and Disjunctive Normal Form (DNF).

---

## What is EQL?

**Entity Query Language (EQL)** is a declarative, relational query language embedded natively in Python. It lets you query Python objects directly. You define variables with finite domains, write conditions, and EQL evaluates these conditions over the Cartesian product of the variables' domains.

EQL has two conceptual layers:
1. **Symbolic representation (syntax):** The query grammar contains variables, conditions on variables, and logical connectives (AND, OR), the query defines *what* you want to know, independently of how it gets computed.
2. **Evaluation backend (semantics):** The default backend enumerates bindings (tuples, where each tuple contains a value for each variable) from finite domains. But the same query syntax can also act as an *integrity constraint* (ensuring that the data is correct or within safe bounds that we specify with conditions) on streaming data (Exercise 4), or as *evidence specification* for a probabilistic model (covered in future tutorials after the probability theory lectures).

EQL is part of **KRROOD** (Knowledge Representation & Reasoning through Object-Oriented Design).  
**Full documentation:** [https://cram2.github.io/cognitive_robot_abstract_machine/krrood/intro.html](https://cram2.github.io/cognitive_robot_abstract_machine/krrood/intro.html)  
**EQL User Guide:** [https://cram2.github.io/cognitive_robot_abstract_machine/krrood/eql/user/index.html](https://cram2.github.io/cognitive_robot_abstract_machine/krrood/eql/user/index.html)

### Domain Model & Fact Base

We define the RoboFleet domain and populate it with data. **Run this cell — all exercises depend on it.**

In [2]:
from dataclasses import dataclass
from typing import List

# --- Domain Model ---

@dataclass
class Zone:
    """A warehouse zone with a name and priority level."""
    name: str
    priority: int  # 1 = low, 2 = medium, 3 = high

@dataclass
class Robot:
    """A warehouse robot with a name, battery level, and assigned zone."""
    name: str
    battery: int      # percentage 0-100
    zone_name: str    # name of the assigned zone

@dataclass
class SensorReading:
    """A position reading from a robot's sensor."""
    robot_name: str
    pos_x: float
    pos_y: float
    timestamp: float

# --- Fact Base ---

zones = [
    Zone("Loading", 3),
    Zone("Storage", 1),
    Zone("Packing", 2),
    Zone("Charging", 1),
]

robots = [
    Robot("Atlas",  85, "Loading"),
    Robot("Bolt",   25, "Storage"),
    Robot("Cora",   60, "Packing"),
    Robot("Dash",   10, "Charging"),
    Robot("Echo",   45, "Loading"),
    Robot("Flux",   90, "Storage"),
]

# EQL imports we'll use throughout
from krrood.entity_query_language.factories import *

print(f"Loaded {len(robots)} robots and {len(zones)} zones.")
print(f"Robots: {[r.name for r in robots]}")
print(f"Zones:  {[z.name for z in zones]}")

Loaded 6 robots and 4 zones.
Robots: ['Atlas', 'Bolt', 'Cora', 'Dash', 'Echo', 'Flux']
Zones:  ['Loading', 'Storage', 'Packing', 'Charging']


---

## Exercise 1: Variables, Domains, and Measurable Spaces

### Concept Review

Recall from the lecture the definition of a **$\sigma$-algebra** (Kolmogorov, 1933):

> Let $E$ be a space of elementary events. A set $\mathfrak{I} \subset 2^E$ is a $\sigma$-algebra if (1) $E \in \mathfrak{I}$, (2) it is closed under set difference, and (3) it is closed under countable union and intersection. The pair $(E, \mathfrak{I})$ is called a **measurable space**.

For a finite set $E$, the **powerset** $2^E$ is always a valid $\sigma$-algebra, giving the measurable space $(E, 2^E)$. The lecture warned that $|2^E|$ grows exponentially.

### Connection to EQL

In EQL, when you write `variable(Robot, domain=robots)`, you are defining a variable whose **domain** is the set `robots` — this is $E$. The measurable space is $(\texttt{robots}, 2^{\texttt{robots}})$. Every possible query result over this single variable is a subset of `robots`, i.e., an element of $2^E$.

### Task 1a

Define an EQL variable `r` over the `robots` domain. Write a query with **no conditions** that retrieves all robots. Verify that the number of results equals $|E|$. Then compute $|2^E|$ and reflect on the exponential blowup.

In [3]:
# Task 1a: Define a variable and query the full domain

r = variable(Robot, domain=robots)

# YOU CODE HERE: Query with no .where(), retrieves the entire domain E
all_robots = ...

E_size = len(all_robots)
powerset_size = 2 ** E_size

print(f"|E| = {E_size} robots")
print(f"|2^E| = {powerset_size} possible subsets (events in the sigma-algebra)")
print()
for robot in all_robots:
    print(f"  {robot.name}: battery={robot.battery}%, zone={robot.zone_name}")

|E| = 6 robots
|2^E| = 64 possible subsets (events in the sigma-algebra)

  Atlas: battery=85%, zone=Loading
  Bolt: battery=25%, zone=Storage
  Cora: battery=60%, zone=Packing
  Dash: battery=10%, zone=Charging
  Echo: battery=45%, zone=Loading
  Flux: battery=90%, zone=Storage


### Task 1b

Now add a condition. Use `.where()` to select only robots with `battery > 50`. The result is a **subset** of $E$ — one particular event in the $\sigma$-algebra. How many robots satisfy this? What fraction of $2^E$ does this single event represent?

In [4]:
# Task 1b: Query with a condition — selecting an event from the sigma-algebra

r = variable(Robot, domain=robots)

# YOUR CODE HERE: Write a query that selects robots with battery > 50
high_battery = ...

print(f"Robots with battery > 50: {[rb.name for rb in high_battery]}")
print(f"This is 1 event out of {2**len(robots)} possible events in 2^E.")

Robots with battery > 50: ['Atlas', 'Cora', 'Flux']
This is 1 event out of 64 possible events in 2^E.


**Reflection:** The variable's domain is the elementary event space $E$. A query condition selects a specific subset — one element of $2^E$. With just 6 robots, there are already 64 possible subsets. The lecture's warning about exponential growth of powersets is already visible at this tiny scale. Product sigma algebras (Exercise 2) will give us a structured way to work with joint spaces without enumerating $2^{E_1 \times E_2}$ explicitly.

---

## Exercise 2: Product Spaces and Conjunction as Intersection

### Concept Review

The **product sigma algebra** (Hunter, 2011) is defined as follows: given measurable spaces $(X, \mathcal{A})$ and $(Y, \mathcal{B})$, the product sigma algebra $\mathcal{A} \times \mathcal{B}$ is the $\sigma$-algebra on $X \times Y$ generated by all measurable rectangles:

$$\mathcal{A} \times \mathcal{B} = \sigma(\{A \times B : A \in \mathcal{A},\ B \in \mathcal{B}\})$$

The key structural result is that **intersection of product sets decomposes component-wise**:

$$(A \times B) \cap (C \times D) = (A \cap C) \times (B \cap D)$$

A **simple event** (measurable rectangle) in this product space is a set of the form $A \times B$ where $A$ and $B$ constrain different variables independently.

### Connection to EQL

When you define two EQL variables `r = variable(Robot, domain=robots)` and `z = variable(Zone, domain=zones)`, a query over both ranges over the **Cartesian product** $\texttt{robots} \times \texttt{zones}$. This product is exactly the product space from the lecture. Adding conditions via `.where(cond_on_r, cond_on_z)` selects a measurable rectangle — the conjunction restricts each variable independently.

### Task 2a

Define two variables — one over `robots`, one over `zones`. Write a query with `set_of(r, z)` and **no conditions** to enumerate the full Cartesian product. Verify that the number of results equals $|\texttt{robots}| \times |\texttt{zones}|$.

In [5]:
# Task 2a: Full Cartesian product — the product space E1 × E2

r = variable(Robot, domain=robots)
z = variable(Zone, domain=zones)

product = ...

print(f"|robots| = {len(robots)}")
print(f"|zones|  = {len(zones)}")
print(f"|robots × zones| = {len(product)}")
print(f"Expected: {len(robots)} × {len(zones)} = {len(robots) * len(zones)}")
print(f"Match: {len(product) == len(robots) * len(zones)}")
print()
print("First 5 pairs:")
for pair in product[:5]:
    print(f"  ({pair[r].name}, {pair[z].name})")

|robots| = 6
|zones|  = 4
|robots × zones| = 24
Expected: 6 × 4 = 24
Match: True

First 5 pairs:
  (Atlas, Loading)
  (Atlas, Storage)
  (Atlas, Packing)
  (Atlas, Charging)
  (Bolt, Loading)


### Task 2b

Now define two sets by adding conditions:
- $A$ = robots with `battery > 30`  
- $B$ = zones with `priority >= 2`

Write a single query with both conditions to get the measurable rectangle $A \times B$. Then compute $A$ and $B$ separately and verify that $|A \times B| = |A| \times |B|$.

In [6]:
# Task 2b: Measurable rectangle via conjunction
r = variable(Robot, domain=robots)
z = variable(Zone, domain=zones)

# Combined query: the measurable rectangle A × B
rectangle = ...

# Individual queries: compute A and B separately
A = ...

B = ...

print(f"|A| = {len(A)}  (robots with battery > 30): {[rb.name for rb in A]}")
print(f"|B| = {len(B)}  (zones with priority >= 2): {[zn.name for zn in B]}")
print(f"|A × B| = {len(rectangle)}")
print(f"|A| × |B| = {len(A) * len(B)}")
print(f"Match: {len(rectangle) == len(A) * len(B)}")

|A| = 4  (robots with battery > 30): ['Atlas', 'Cora', 'Echo', 'Flux']
|B| = 2  (zones with priority >= 2): ['Loading', 'Packing']
|A × B| = 8
|A| × |B| = 8
Match: True


**Reflection:** The `.where()` clause with conditions on different variables produces a measurable rectangle in the product sigma algebra. The conjunction decomposes component-wise — each variable is filtered independently. This is why the product sigma algebra is computationally efficient: instead of enumerating $2^{|E_1 \times E_2|}$ subsets, we work with structured rectangles $A \times B$ where $A \subseteq E_1$ and $B \subseteq E_2$.

---

## Exercise 3: Disjunction, DNF, and Complement

### Concept Review

The lecture established two key ideas:

1. **Composite sets as DNF.** A **composite set** is a union of simple sets (measurable rectangles). This is equivalent to **Disjunctive Normal Form** in logic — a disjunction (OR) of conjunctions (AND) of variable assignments:

$$\underbrace{(\text{Utensil} \in \{\text{bowl}\} \wedge \text{Color} \in \{\text{blue}\})}_{\text{simple set 1}} \vee \underbrace{(\text{Utensil} \in \{\text{cup}\} \wedge \text{Color} \in \{\text{red}\})}_{\text{simple set 2}}$$

2. **Efficient complement.** The complement of a product set decomposes as:

$$(A \times B)^c = (A^c \times B) \cup (A \times B^c)$$

This avoids the naive $2^n - 1$ term expansion.

### Connection to EQL

- `or_()` produces a **union of conjunctive queries (UCQ)** — exactly a composite set in DNF form. Each disjunct (each operand of the `or_()` operation) is one "simple set" (one conjunctive query), and the full `or_()` is the union.
- `not_()` implements **Negation as Failure** — it is true if the condition inside it is false. Thus, it only allows elements of the complement to pass through.
- `and_()` is the **conjunction** operator it takes conditions as arguments, it is equivalent to a simple set in DNF.

### Task 3a — Composite sets via `or_()`

Build a composite event that selects robot–zone pairs satisfying **either** of these conditions:
- Robots with `battery > 70` assigned to zones with `priority == 3` (high-battery robots in critical zones)
- Robots with `battery < 20` assigned to zones with `priority == 1` (low-battery robots in low-priority zones)

This is a union of two measurable rectangles — a composite set in DNF.

In [7]:
# Task 3a: Composite set (UCQ) via or_() — DNF in action

r = variable(Robot, domain=robots)
z = variable(Zone, domain=zones)

# YOUR CODE HERE: Build the composite event as a disjunction of two conjunctions
composite = ...

print(f"Composite event has {len(composite)} pairs (union of two rectangles):")
for pair in composite:
    print(f"  Robot {pair[r].name} (battery={pair[r].battery}) × Zone {pair[z].name} (priority={pair[z].priority})")

Composite event has 4 pairs (union of two rectangles):
  Robot Atlas (battery=85) × Zone Loading (priority=3)
  Robot Dash (battery=10) × Zone Storage (priority=1)
  Robot Dash (battery=10) × Zone Charging (priority=1)
  Robot Flux (battery=90) × Zone Loading (priority=3)


### Task 3b — Complement via `not_()` and the complement decomposition theorem

The lecture showed that $(A \times B)^c = (A^c \times B) \cup (A \times B^c) \cup (A^c \times B^c)$.

Let's verify this with EQL. Define:
- $A$ = robots with `battery > 50`
- $B$ = zones with `priority >= 2`

Compute:
1. The **rectangle** $A \times B$ using a conjunction.
2. The **complement** $(A \times B)^c$ using `not_()` on both conditions.
3. The **decomposition** $(A^c \times B) \cup (A \times B^c) \cup (A^c \times B^c)$ manually as an `or_()` of three rectangles.

Verify that (2) and (3) produce the same number of results, and that the rectangle plus its complement cover the entire product space.

In [12]:
# Task 3b: Complement and the complement decomposition theorem

# (1) The rectangle A × B
r = variable(Robot, domain=robots)
z = variable(Zone, domain=zones)
rectangle = ...

# (2) The complement via not_(): everything NOT in (A × B)
# This means: not (battery > 50 AND priority >= 2)
complement_naf = ...

# (3) Decomposition: (A^c × B) ∪ (A × B^c) U (A^c x B^c)
complement_manual = ...

# (4) Efficient Decomposition: (A^c × Whole of B) ∪ (A × B^c)
complement_efficient = ...

# Full product space for reference
total = len(robots) * len(zones)

print(f"Full product space: {total} pairs")
print(f"|A × B| (rectangle):           {len(rectangle)}")
print(f"|(A × B)^c| via not_():        {len(complement_naf)}")
print(f"|(A^c×B) ∪ (A×B^c) U (A^c×B^c)| (manual): {len(complement_manual)}")
print(f"|(A^c×B_whole) ∪ (A×B^c)| (efficient): {len(complement_efficient)}")
print()
print(f"Rectangle + complement = {len(rectangle) + len(complement_naf)} (should be {total})")
print(f"NAF complement matches manual decomposition: {len(complement_naf) == len(complement_manual)}")
print(f"Efficient decomposition matches manual decomposition: {len(complement_efficient) == len(complement_manual)}")

Full product space: 24 pairs
|A × B| (rectangle):           6
|(A × B)^c| via not_():        18
|(A^c×B) ∪ (A×B^c) U (A^c×B^c)| (manual): 18
|(A^c×B_whole) ∪ (A×B^c)| (efficient): 18)

Rectangle + complement = 24 (should be 24)
NAF complement matches manual decomposition: True
Efficient decomposition matches manual decomposition: True




**Reflection:** `or_()` is DNF — it builds composite sets as unions of simple sets (rectangles). `not_()` implements complement via Negation as Failure. The efficient complement theorem from the lecture tells you how to decompose a complement into a *small* union of rectangles instead of listing individual elements.

---

## Exercise 4: From Discrete Domains to Continuous Constraints

### Concept Review

The lecture moved from finite discrete domains to continuous domains ($\mathbb{R}^d$), introducing:
- **Intervals** as the building blocks for describing subsets of $\mathbb{R}$.
- **Simple events** in a product sigma algebra over $\mathbb{R}^2$ as **rectangles**: e.g., $x \in [2, 3) \times y \in [10, 15)$.
- The fact that product sigma algebras can only represent **hyper-rectangles**, not circles or triangles (which involve dependent variable constraints).
- **Outer and inner box approximations** as a way to approximate non-rectangular shapes with unions of rectangles.

### Connection to EQL — Queries as Integrity Constraints

EQL's default backend enumerates finite domains. But what if the domain is a **stream of sensor data**? The same EQL query can be wrapped in a function that receives a chunk of data, passes it as the domain, and evaluates the query. The conditions become **integrity constraints** that filter the stream — the query defines a measurable set, and the backend checks membership.

This is the key two-layer insight: the **syntax** defines the event (the rectangle $x \in [2, 3) \times y \in [10, 15)$), and the **backend** determines whether we enumerate or filter.

### Task 4

Simulate a stream of robot position sensor readings. Define a function that wraps an EQL query acting as a spatial constraint (a measurable rectangle in $\mathbb{R}^2$). Feed the stream through the query and observe which readings fall inside the rectangle.

The rectangle represents a **safe operating region** in the warehouse: $x \in [2.0, 5.0]$ and $y \in [1.0, 4.0]$.

In [13]:
import random
random.seed(42)

# Simulate a stream of 20 sensor readings with positions in [0, 7] × [0, 7]
sensor_stream = [
    SensorReading(
        robot_name=random.choice(["Atlas", "Bolt", "Cora"]),
        pos_x=round(random.uniform(0, 7), 2),
        pos_y=round(random.uniform(0, 7), 2),
        timestamp=round(t * 0.5, 1)
    )
    for t in range(20)
]

print(f"Generated {len(sensor_stream)} sensor readings.")
for s in sensor_stream[:5]:
    print(f"  t={s.timestamp}: {s.robot_name} at ({s.pos_x}, {s.pos_y})")
print("  ...")

Generated 20 sensor readings.
  t=0.0: Cora at (0.78, 5.19)
  t=0.5: Atlas at (1.56, 5.16)
  t=1.0: Cora at (5.18, 3.82)
  t=1.5: Cora at (2.95, 0.21)
  t=2.0: Atlas at (1.63, 4.21)
  ...


In [14]:
# Task 4: EQL query as an integrity constraint on streaming data

def filter_safe_region(readings, x_min=2.0, x_max=5.0, y_min=1.0, y_max=4.0):
    """
    Filter sensor readings that fall within the safe operating rectangle.
    
    The EQL query defines a measurable rectangle in R^2:
        x ∈ [x_min, x_max] × y ∈ [y_min, y_max]
    
    The readings list is the domain — swapped in at call time.
    """
    sensor_readings = variable(SensorReading, domain=readings)
    # YOUR CODE HERE: Define the EQL query for the safe region.
    safe_regions = ...
    return safe_regions.tolist()


# Apply the constraint to the stream
safe_readings = filter_safe_region(sensor_stream)
unsafe_readings = [s for s in sensor_stream if s not in safe_readings]

print(f"Safe region: x ∈ [2.0, 5.0] × y ∈ [1.0, 4.0]")
print(f"Total readings: {len(sensor_stream)}")
print(f"Inside safe region:  {len(safe_readings)}")
print(f"Outside safe region: {len(unsafe_readings)}")
print()
print("Readings inside the rectangle:")
for s in safe_readings:
    print(f"  t={s.timestamp}: {s.robot_name} at ({s.pos_x}, {s.pos_y}) ✓")

Safe region: x ∈ [2.0, 5.0] × y ∈ [1.0, 4.0]
Total readings: 20
Inside safe region:  2
Outside safe region: 18

Readings inside the rectangle:
  t=3.0: Cora at (2.94, 3.14) ✓
  t=4.0: Atlas at (4.89, 2.38) ✓


**Reflection:** The EQL query syntax — `s.pos_x >= 2.0, s.pos_x <= 5.0, s.pos_y >= 1.0, s.pos_y <= 4.0` — defines a **measurable rectangle** in $\mathbb{R}^2$, exactly like the simple events from lecture slides 35–36. The function `filter_safe_region` swaps in a new domain on every call. The query doesn't change — only the data does. This is EQL's two-layer architecture in action: the symbolic layer defines *what* the safe region is, and the backend evaluates membership.

The lecture's observation that product sigma algebras can only represent **hyper-rectangles** is True, but with EQL you can express arbitrary constraints: if the safe region were a circle ($x^2 + y^2 \leq r^2$), a single rectangle condition wouldn't suffice. You'd need the **inner/outer box approximation** strategy from the lecture — a union of rectangles (a composite set via `or_()`) to approximate the circular boundary.

In future tutorials (after the probability theory lectures), the same EQL query syntax will serve as **evidence specification** for a probabilistic model — the conditions become observations, and the model returns distributions rather than filtered lists.

---

## Summary: Algebra ↔ EQL Correspondence

| Algebra Concept | EQL Construct                                                                     | Exercise |
|---|-----------------------------------------------------------------------------------|---|
| Elementary event space $E$ | `domain=robots` in `variable(Robot, domain=robots)`                               | 1 |
| Measurable space $(E, 2^E)$ | All possible query results over one variable                                      | 1 |
| Product space $E_1 \times E_2$ | Multi-variable query with `set_of(r, z)`                                          | 2 |
| Simple set (measurable rectangle) | Conjunctive query: `.where(cond1, cond2)`                                         | 2 |
| Intersection theorem | Conjunction over independent variables decomposes per-variable                    | 2 |
| Composite set (union of simple sets) | UCQ via `or_()`                                                                   | 3 |
| Disjunctive Normal Form (DNF) | `or_(and_(...), and_(...))`                                                       | 3 |
| Complement | `not_()` (Negation as Failure)                                                    | 3 |
| Efficient complement theorem | Manual decomposition into `or_(A^c × B_whole, A × B^c)`                           | 3 |
| Intervals / rectangles in $\mathbb{R}^2$ | Inequality conditions as integrity constraints                                    | 4 |
| Inner/outer box approximation | Union of rectangle queries (composite set) to approximate non-rectangular regions | 4 (discussion) |

### What's Next

The algebra from this lecture provides the **event structure** on which probability measures are defined. In the coming lectures on probability theory, you will see how probability functions are assigned to these events. EQL will reappear as a way to specify **evidence** for probabilistic queries — the same query that filtered sensor readings in Exercise 4 will instead condition a probabilistic model. Stay tuned.